# CardioScore — held-out Cardio PyMEA validation

Two modes: `PREPARE` creates and auto-downloads a blinded annotation package; `SCORE` verifies that package, scores 1–2 completed annotation CSVs with the frozen v1 criteria, and auto-downloads the complete result bundle.

For PREPARE, put `Day9__Nov1_130pm.txt` in Google Drive. Preferably also provide the exact earlier detector-development `sheet_manifest.json`. Without that exact manifest, the notebook reconstructs the historical default 20-window batch and permanently labels the result provisional/exploratory rather than confirmatory.


In [ ]:
#@title 1) Configuration
MODE = "PREPARE"  #@param ["PREPARE", "SCORE"]
VALIDATION_CODE_SHA = "39767da5144946f73717b34c2ba7f1e63766afef"
PROTOCOL_COMMIT = "7c09d907f5c4949ef25209465de56df44a855d99"
CRITERIA_REL = "validation/signal_validation_criteria_v1.yaml"
RAW_RECORDING_PATH = ""  #@param {type:"string"}
DEV_MANIFEST_PATH = ""  #@param {type:"string"}
RAW_BASENAME = "Day9__Nov1_130pm.txt"
N_HELDOUT = 40
WINDOW_S = 10.0
HELDOUT_SEED = 20261004
WORK_ROOT = "/content/cardioscore_pymea_validation"
REPO_DIR = WORK_ROOT + "/repo"


In [ ]:
#@title 2) Locked setup, focused tests, synthetic benchmark
import os, sys, json, shutil, subprocess, hashlib, platform, zipfile, importlib.util
from pathlib import Path
from datetime import datetime, timezone

WORK = Path(WORK_ROOT)
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)

def run(cmd, cwd=None):
    print("+", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, check=True)

run(["git","clone","-q","https://github.com/Virelion-Biotech/Virelion-CardioScore.git",REPO_DIR])
run(["git","checkout","-q",PROTOCOL_COMMIT], REPO_DIR)

d = subprocess.run([
    "git","diff","--exit-code",VALIDATION_CODE_SHA,PROTOCOL_COMMIT,"--",
    "virelion_cardioscore",
    "scripts/validation/pymea_beat_validation.py",
    "scripts/validation/synthetic_signal_benchmark.py",
    "scripts/validation/inspect_raw_source.py"
], cwd=REPO_DIR, text=True, capture_output=True)
if d.returncode:
    print(d.stdout, d.stderr)
    raise RuntimeError("Protocol commit changes executable validation code.")

run([sys.executable,"-m","pip","install","-q","-e",".[dev,mixed]"], REPO_DIR)
import yaml
criteria_path = Path(REPO_DIR)/CRITERIA_REL
criteria_bytes = criteria_path.read_bytes()
criteria_sha = hashlib.sha256(criteria_bytes).hexdigest()
criteria = yaml.safe_load(criteria_bytes)
assert criteria["status"] == criteria["beat_detection"]["status"] == "frozen"
assert criteria["validation_code_sha"] == VALIDATION_CODE_SHA

EVIDENCE = WORK/"environment_evidence"; EVIDENCE.mkdir()
focused = ["tests/test_mcs_ascii.py","tests/test_signal_accuracy.py","tests/test_pymea_beat_validation.py","tests/test_stv_semantics.py"]
p = subprocess.run([sys.executable,"-m","pytest","-q",*focused], cwd=REPO_DIR, text=True, capture_output=True)
(EVIDENCE/"focused_pytest.txt").write_text(p.stdout+"\n"+p.stderr)
print(p.stdout)
if p.returncode: raise RuntimeError("Focused validation tests failed.")
run([sys.executable,str(Path(REPO_DIR)/"scripts/validation/synthetic_signal_benchmark.py"),
     "--out",str(EVIDENCE/"synthetic_benchmark"),"--seeds","5"], REPO_DIR)

setup_info = {
 "validation_code_sha":VALIDATION_CODE_SHA, "protocol_commit":PROTOCOL_COMMIT,
 "criteria_sha256":criteria_sha, "generated_at_utc":datetime.now(timezone.utc).isoformat(),
 "python":sys.version, "platform":platform.platform()
}


In [ ]:
#@title 3A) PREPARE — source, exclusions, blind package
if MODE != "PREPARE":
    print("Skipping PREPARE")
else:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)

    raw = Path(RAW_RECORDING_PATH) if RAW_RECORDING_PATH.strip() else None
    if raw is None or not raw.is_file():
        matches = list(Path("/content/drive/MyDrive").rglob(RAW_BASENAME))
        if len(matches) != 1:
            raise RuntimeError("Expected exactly one "+RAW_BASENAME+" in Drive; found: "+str(matches))
        raw = matches[0]
    dev_manifest = Path(DEV_MANIFEST_PATH) if DEV_MANIFEST_PATH.strip() else None
    if dev_manifest and not dev_manifest.is_file(): raise FileNotFoundError(dev_manifest)

    PREP = WORK/"prepare"; PREP.mkdir()
    run([sys.executable,str(Path(REPO_DIR)/"scripts/validation/inspect_raw_source.py"),
         str(raw),"--out",str(PREP/"source_inventory.json")], REPO_DIR)

    sys.path.insert(0, REPO_DIR)
    from virelion_cardioscore.io.mcs_ascii import sniff_layout
    layout = sniff_layout(raw)
    h = hashlib.sha256()
    with raw.open("rb") as f:
        for block in iter(lambda:f.read(8<<20), b""): h.update(block)
    raw_sha = h.hexdigest()

    tool_path = Path(REPO_DIR)/"scripts/validation/pymea_beat_validation.py"
    spec = importlib.util.spec_from_file_location("pymea_locked", tool_path)
    pymea = importlib.util.module_from_spec(spec); spec.loader.exec_module(pymea)

    exact = False
    if dev_manifest:
        dev = json.loads(dev_manifest.read_text())
        if dev.get("source_sha256") != raw_sha: raise RuntimeError("Development manifest source hash mismatch.")
        exact = True
    else:
        dev = pymea.make_sheet(raw, PREP/"reconstructed_dev", n_windows=20, window_s=10.0,
                               seed=20260921, duration_s=None, exclude=[])

    exclusions = [(str(w["electrode"]),float(w["start_s"]),float(w["start_s"])+float(w["window_s"]))
                  for w in dev["windows"]]
    held = PREP/"heldout_annotation"
    manifest = pymea.make_sheet(raw, held, n_windows=N_HELDOUT, window_s=WINDOW_S,
                                seed=HELDOUT_SEED, duration_s=None, exclude=exclusions)

    for w in manifest["windows"]:
        a=float(w["start_s"]); b=a+float(w["window_s"])
        if any(w["electrode"]==e and a<z and b>y for e,y,z in exclusions):
            raise RuntimeError("Held-out overlap detected.")

    evidence_state = "confirmatory_heldout_candidate" if exact else "provisional_exploratory_reconstructed_development_exclusions"
    pkg = PREP/"annotation_package"
    shutil.copytree(held, pkg/"heldout_annotation")
    shutil.copy2(criteria_path,pkg/"signal_validation_criteria_v1.yaml")
    shutil.copy2(PREP/"source_inventory.json",pkg/"source_inventory.json")
    shutil.copytree(EVIDENCE,pkg/"environment_evidence")
    (pkg/"development_exclusions.json").write_text(json.dumps({
        "source_sha256":raw_sha,"exact_development_manifest_supplied":exact,
        "development_manifest":dev,"excluded_regions":[list(x) for x in exclusions]
    },indent=2)+"\n")
    protocol = {**setup_info,"evidence_state":evidence_state,
        "confirmatory_eligible_if_annotations_complete":exact,
        "source_filename":raw.name,"source_sha256":raw_sha,"source_bytes":raw.stat().st_size,
        "fs_hz":float(layout.fs_hz),"heldout_seed":HELDOUT_SEED,
        "heldout_n_windows":N_HELDOUT,"heldout_window_s":WINDOW_S,
        "frozen_acceptance":criteria["beat_detection"]}
    (pkg/"protocol.json").write_text(json.dumps(protocol,indent=2)+"\n")
    (pkg/"ANNOTATE_ME.txt").write_text(
        "Fill heldout_annotation/annotation_sheet.csv blindly. For every row set done=1; "
        "beat_times_s are semicolon-separated seconds relative to window start; unsure=1 only if unreadable. "
        "Do not inspect CardioScore detections before annotation.\n"
    )

    zpath = Path("/content")/f"cardioscore_pymea_annotation_package_{HELDOUT_SEED}.zip"
    with zipfile.ZipFile(zpath,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for q in pkg.rglob("*"):
            if q.is_file(): z.write(q,arcname=str(q.relative_to(pkg)))
    print("Evidence state:", evidence_state)
    print("Annotation package:", zpath)
    files.download(str(zpath))


In [ ]:
#@title 3B) SCORE — upload package + 1–2 completed annotation CSVs
if MODE != "SCORE":
    print("Skipping SCORE")
else:
    from google.colab import files
    uploaded = files.upload()
    up = WORK/"uploads"; up.mkdir()
    paths=[]
    for name,data in uploaded.items():
        q=up/Path(name).name; q.write_bytes(data); paths.append(q)
    zips=[q for q in paths if q.suffix.lower()==".zip"]
    csvs=[q for q in paths if q.suffix.lower()==".csv"]
    if len(zips)!=1 or not (1<=len(csvs)<=2):
        raise RuntimeError("Upload exactly one annotation-package ZIP plus 1–2 annotation CSVs.")
    package_zip=zips[0]; annotation_csvs=sorted(csvs)


In [ ]:
#@title 4B) SCORE — verify, score, export, auto-download
if MODE == "SCORE":
    import pandas as pd
    from google.colab import files
    SCORE=WORK/"score"; pkg=SCORE/"package"; out=SCORE/"results"
    pkg.mkdir(parents=True); out.mkdir()
    with zipfile.ZipFile(package_zip) as z:
        bad=[n for n in z.namelist() if n.startswith(("/", "\\")) or ".." in Path(n).parts]
        if bad: raise RuntimeError("Unsafe ZIP paths: "+str(bad))
        z.extractall(pkg)

    protocol=json.loads((pkg/"protocol.json").read_text())
    if protocol["validation_code_sha"]!=VALIDATION_CODE_SHA or protocol["protocol_commit"]!=PROTOCOL_COMMIT:
        raise RuntimeError("Package protocol mismatch.")
    pc=pkg/"signal_validation_criteria_v1.yaml"
    if hashlib.sha256(pc.read_bytes()).hexdigest()!=criteria_sha:
        raise RuntimeError("Frozen criteria hash mismatch.")

    anns=[]
    for q in annotation_csvs:
        dst=out/q.name; shutil.copy2(q,dst); anns.append(dst)

    tool_path=Path(REPO_DIR)/"scripts/validation/pymea_beat_validation.py"
    spec=importlib.util.spec_from_file_location("pymea_score_locked",tool_path)
    pymea=importlib.util.module_from_spec(spec); spec.loader.exec_module(pymea)
    results=pymea.score(pkg/"heldout_annotation",anns,
        float(criteria["beat_detection"]["tolerance_ms"])/1000.0,pc,pymea.FilterConfig())

    exact=bool(protocol["confirmatory_eligible_if_annotations_complete"])
    cp=results.get("criteria",{}).get("passed")
    results["evidence_interpretation"]={
      "protocol_evidence_state":protocol["evidence_state"],
      "exact_development_exclusions":exact,
      "criteria_confirmatory":results.get("criteria",{}).get("confirmatory"),
      "criteria_passed":cp,
      "confirmatory_interpretation_allowed":bool(exact and cp is not None),
      "confirmatory_passed":cp if exact else None,
      "reason":None if exact else "Exact earlier development sheet_manifest.json was not supplied; result is provisional/exploratory."
    }
    (out/"pymea_heldout_validation_results.json").write_text(json.dumps(results,indent=2,default=float)+"\n")

    rows=[]
    for ann,m in results["vs_annotator"].items():
        for wid,wm in m["per_window"].items(): rows.append({"annotator":ann,"window_id":wid,**wm})
    pd.DataFrame(rows).to_csv(out/"per_window_detection_counts.csv",index=False)
    pd.DataFrame(results["qc"]["windows"]).to_csv(out/"qc_windows.csv",index=False)
    pd.DataFrame([{"annotator":a,**{k:m[k] for k in ["tp","fp","fn","precision","recall","f1","timing_median_abs_error_ms","timing_p95_abs_error_ms","n_unsure_windows_excluded"]}}
                  for a,m in results["vs_annotator"].items()]).to_csv(out/"metric_summary.csv",index=False)

    for src,name in [(pkg/"protocol.json","protocol.json"),(pc,"signal_validation_criteria_v1.yaml"),
                     (pkg/"source_inventory.json","source_inventory.json"),
                     (pkg/"development_exclusions.json","development_exclusions.json"),
                     (pkg/"heldout_annotation"/"sheet_manifest.json","heldout_sheet_manifest.json")]:
        shutil.copy2(src,out/name)
    if (pkg/"environment_evidence").exists():
        shutil.copytree(pkg/"environment_evidence",out/"environment_evidence")

    rm={"generated_at_utc":datetime.now(timezone.utc).isoformat(),
        "validation_code_sha":VALIDATION_CODE_SHA,"protocol_commit":PROTOCOL_COMMIT,
        "evidence_interpretation":results["evidence_interpretation"],"artifacts":{}}
    for q in sorted(out.rglob("*")):
        if q.is_file(): rm["artifacts"][str(q.relative_to(out))]={"bytes":q.stat().st_size,"sha256":hashlib.sha256(q.read_bytes()).hexdigest()}
    (out/"RESULTS_MANIFEST.json").write_text(json.dumps(rm,indent=2)+"\n")

    zpath=Path("/content/cardioscore_pymea_heldout_validation_results.zip")
    with zipfile.ZipFile(zpath,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for q in out.rglob("*"):
            if q.is_file(): z.write(q,arcname=str(q.relative_to(out)))
    print(json.dumps(results["evidence_interpretation"],indent=2))
    for a,m in results["vs_annotator"].items():
        print(a,"F1",round(m["f1"],4),"precision",round(m["precision"],4),"recall",round(m["recall"],4))
    print("Send this ZIP back to ChatGPT:",zpath)
    files.download(str(zpath))


After SCORE, send back only `cardioscore_pymea_heldout_validation_results.zip`. If the frozen detector passes, commit results/provenance. If it fails and the detector is changed, this dataset becomes exposed and a new untouched validation set is required for a later confirmatory claim.